# Practical Gassmann Fluid Substitution in Sand/Shale Sequences

**Reference:** Simm, R. (2007). *Practical rock physics*. First Break, Vol 25, December 2007.

**Well:** WELL-2, Glitne Field, Norway (Heimdal Formation — oil field)
**Data:** Vp (km/s), Vs (km/s), RHOB (g/cc), GR (GAPI), NPHI

---

## Workflow Overview — Simm's 6-Step Adaptive Approach

| Step | Description |
|------|-------------|
| 1 | Load well logs (Vp, Vs, density, GR) and assign facies labels |
| 2 | Compute Vsh from GR; effective porosity from density log |
| 3 | Mineral modulus K₀ via Voigt-Reuss-Hill (VRH) mixing (quartz + shale) |
| 4 | Gassmann inversion → dry-rock parameters (Kd/K₀, Kφ/K₀) |
| 5 | QC on Kd/K₀ vs φ crossplot; fit adaptive polynomial dry-rock trend |
| 6 | Forward Gassmann substitution: brine → oil  AND  brine → gas |

The key innovation in Simm's approach is **step 5**: instead of using the raw inverted
dry-frame modulus for every sample, a polynomial trend is fitted to the cleanest
reference sands and applied **selectively** to intermediate (silty) lithologies.
This prevents noisy inversion results from propagating into the forward substitution.

## 1. Imports

Standard scientific Python stack. `%matplotlib inline` renders figures directly
in the notebook rather than writing them to disk. The `glob` module is used to
discover per-facies data files by wildcard pattern.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
from rockphys import *


## 2. Mineral and Fluid Constants

These constants define the **end-member physical properties** for all rock physics
calculations that follow:

- **Mineral properties** (quartz, clay/shale): elastic moduli (bulk K, shear G in GPa)
  and grain density (g/cc) from Simm (2007) and `info_params.txt`.
- **Fluid properties** (brine, gas): bulk modulus and density at reservoir conditions.
  Oil properties are computed from Batzle-Wang (1992) in the next section.
- **Reservoir conditions**: temperature, pressure, API gravity, and GOR feed the
  live-oil calculation. Depth markers (Top Heimdal, OWC) annotate every log plot.

These values anchor the VRH mineral mixing and the Gassmann substitution
throughout the entire workflow.

## 3. Data Loading

The well data are stored as whitespace-delimited ASCII text files — one master log
file (`well_2.txt`) and one file per facies population (`well2_clnSand.txt`, etc.).

**`_parse_txt`** — low-level parser that skips comment/header lines, returning a
DataFrame with columns `[depth, Vp, Vs, rho, GR, nphi]`.

**`load_well`** — reads the full well log for WELL-2.

**`load_facies`** — reads each per-facies file and collects the depths belonging
to that lithology class. Six facies are recognised:
*Clean Sand, Cemented Sand, Silty Sand 1, Silty Sand 2, Silty Shale, Shale.*

**`assign_facies`** — depth-matches the facies populations back onto the main well
DataFrame, adding a `facies` column used for colour-coding every subsequent plot.

### Execute: Load Well Data and Assign Facies

Parse the master log file then overlay the per-facies depth populations to
label every depth sample with its lithology class. Printing the facies counts
is a quick sanity-check that all six classes loaded correctly.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data')

print("[1] Loading well data ...")
well = load_well(DATA_DIR)
print(f"    {len(well)} samples, depth {well['depth'].min():.0f}–{well['depth'].max():.0f} m")
print(f"    Columns: {list(well.columns)}")

print("\n[2] Assigning facies ...")
facies_depths = load_facies(DATA_DIR)
well = assign_facies(well, facies_depths)
print(well['facies'].value_counts().to_string(header=False))

well.head()

## 4. Batzle-Wang Live-Oil Fluid Properties

Accurate fluid substitution requires the **in-situ bulk modulus of the pore fluid**
at reservoir temperature and pressure. Batzle & Wang (1992) provide empirical
equations derived from laboratory measurements on a wide range of fluids.

**For live oil the procedure is:**
1. Dead-oil surface density from API gravity
2. Formation volume factor B₀ (accounts for dissolved gas swelling the oil)
3. Live-oil density at reservoir T and P
4. Dead-oil P-wave velocity at T (°C) and P (MPa) — B-W Eq. 20
5. Live-oil velocity via dissolved-gas correction (substitute live density into Eq. 20)
6. K_oil = ρ_live × (Vp_live / 1000)²  [GPa]

> **Unit note:** GOR for this well is in metric (sm³/sm³). B-W equations use
> imperial (scf/stb), so GOR is converted by × 5.615 inside the function.

### Execute: Compute Oil Properties

Apply the Batzle-Wang equations at the Glitne Field reservoir conditions.
The B-W live density can differ from the measured reservoir value for high-GOR
oils, so we use the **given density** (0.78 g/cc) to compute K_oil while still
relying on the B-W velocity equation.

In [ ]:
print("[0] Computing oil properties (Batzle-Wang 1992) ...")
K_OIL_bw, rho_oil_bw, Vp_oil_mps = batzle_wang_oil(
    RESERVOIR_TEMP, RESERVOIR_P, OIL_API, OIL_GOR)

# Use given density (0.78 g/cc) because B-W live density is unreliable
# for this high-GOR case; keep B-W velocity.
K_OIL = RHO_OIL * (Vp_oil_mps / 1000.0)**2

print(f"  B-W live density:     {rho_oil_bw:.3f} g/cc  (given: {RHO_OIL:.2f} g/cc)")
print(f"  B-W live velocity:    {Vp_oil_mps:.0f} m/s")
print(f"  K_oil (adopted):      {K_OIL:.3f} GPa  [= {RHO_OIL:.2f} × ({Vp_oil_mps/1000:.3f})²]")
print(f"  K_gas:                {K_GAS:.3f} GPa  |  K_brine: {K_BR:.3f} GPa")
print()
print("Fluid stiffness ranking (stiffer fluids → higher Vp after substitution):")
print(f"  Gas ({K_GAS}) << Oil ({K_OIL:.3f}) < Brine ({K_BR})  [GPa]")

## 5. Core Rock Physics Functions

Four fundamental functions underpin the entire substitution workflow:

### Voigt-Reuss-Hill (VRH) Mixing
`vrh(f, M1, M2)` — the arithmetic mean of the Voigt upper bound and Reuss lower
bound for a two-component mineral mixture. Used to compute the mineral frame
modulus K₀ (and G₀) as a function of clay volume Vsh.

### Gassmann Inversion
`gassmann_inv` — extracts the normalised dry-frame bulk modulus Kd/K₀ from the
saturated log-derived modulus. Closed-form inverse (Simm 2007 Eq. 4):
$$x = \frac{y\beta - 1}{y + \beta - 2}, \quad \beta = \frac{\phi K_0}{K_{fl}} + 1 - \phi, \quad y = \frac{K_{sat}}{K_0}$$

### Gassmann Forward
`gassmann_fwd` — predicts the new saturated modulus after replacing the pore fluid.
Shear modulus **μ is fluid-independent** — only Ksat changes.

### Poisson's Ratio
`poisson(Vp, Vs)` — the most sensitive seismic fluid indicator; used in crossplots.

## 6. Rock Physics Pipeline — Steps 2–4

`compute_rock_physics` chains the first four computational steps of Simm's
workflow into a single function that enriches the well DataFrame:

| Output column | Derivation |
|---------------|-----------|
| `Vsh` | Linear GR normalisation between 5th / 95th percentile end-points |
| `phi` | Density-porosity: (ρ_mineral − ρ_log) / (ρ_mineral − ρ_brine) |
| `K0`, `G0` | VRH mineral frame moduli (quartz + shale mixture) |
| `mu`, `Ksat` | Saturated moduli from log velocities and density |
| `Kd_K0` | Gassmann inversion — assumes brine saturation throughout |
| `Kphi_K0` | Normalised pore stiffness = Kd/K₀ / (1 − Kd/K₀) |
| `PR`, `AI`, `VpVs` | Derived seismic quantities |

> **Unit note:** ρ [g/cc] × V² [km/s]² = GPa — no conversion needed.

### Execute: Compute Rock Physics

Run the pipeline on the loaded well. The Kd/K₀ statistics give an early
indication of data quality — negative values in shaly or low-porosity zones
are expected when the brine-saturation assumption is locally violated.

In [ ]:
print("[3] Computing rock physics ...")
well = compute_rock_physics(well)

print(f"  Porosity range:   {well['phi'].min():.3f} – {well['phi'].max():.3f} v/v")
print(f"  Kd/K0  P5–P95:   {well['Kd_K0'].quantile(0.05):.3f} – {well['Kd_K0'].quantile(0.95):.3f}")
print(f"  (negative Kd/K0 expected in shaly / low-φ zones)")
print()
# Facies summary of key properties
print(well.groupby('facies')[['phi', 'Vp', 'Vs', 'Kd_K0']].mean().round(3).to_string())

## 7. Plotting Utilities

### Facies colour scheme
Each lithology class is assigned a distinct colour used consistently across every
figure. Warm yellows/oranges = sands; greens = silty sands; greys = shaly facies.

### `_shade_facies`
Adds translucent colour bands behind any depth-track axis to indicate facies
intervals — a standard wireline log convention.

### `_add_reservoir_markers`
Draws dashed horizontal lines at Top Heimdal (2153 m) and the OWC (2183 m) on
every axis in a figure. These are the key boundaries for the Glitne reservoir.

## 8. Dry Rock Trend Fitting — Simm's Adaptive Conditioning (Step 5)

This is the **central methodological contribution** of Simm (2007).

**Problem:** Gassmann inversion of shaley or poorly consolidated sands often
produces Kd/K₀ values that are negative or otherwise unphysical — not because
the physics is wrong, but because Vs logs in mixed lithologies are noisy and
the 100% brine-saturation assumption may be locally invalid.

**Solution:** Fit a 2nd-order polynomial to the **clean-sand** population in
Kd/K₀ vs φ space, anchored at the mineral point (φ = 0, Kd/K₀ = 1):
$$K_d/K_0 = a\phi^2 + b\phi + 1$$
This trend is then applied **only to silty sands** (the problematic intermediate
lithologies). Clean sands, cemented sands, and shales keep their raw inverted
Kd/K₀ values.

### Execute: Fit the Dry-Rock Trend

Run the least-squares fit and print the coefficients. The polynomial should
pass through (φ=0, Kd/K₀=1) by construction and curve downward with increasing
porosity — stiffer at lower porosity (cemented) and more compliant at higher
porosity (clean loose sand).

In [ ]:
print("[4] Fitting conditioned dry-rock trend ...")
coeffs = fit_dry_rock_trend(well)
print(f"  Kd/K0 = {coeffs[0]:.3f}·φ² + {coeffs[1]:.3f}·φ + {coeffs[2]:.3f}")
print("  Conditioning strategy:")
print("    • Clean Sand & Cemented Sand: raw Kd/K0 (these are the reference facies)")
print("    • Silty Sand 1 & 2:           conditioned polynomial trend")
print("    • Shales:                     raw Kd/K0 (too different from sand trend)")

### Figure 2 — Kd/K₀ vs Porosity Template

This is the **central QC diagnostic** from Simm (2007), Figs 5 & 8.

- **X-axis:** effective porosity φ_e
- **Y-axis:** normalised dry bulk modulus Kd/K₀
  (0 = fully compliant framework; 1 = rigid mineral)
- **Horizontal dashed lines:** contours of constant pore stiffness
  Kφ/K₀ = 0.1, 0.2, … 0.5 (horizontal because Kφ/K₀ depends only on Kd/K₀)
- **Black star:** mineral point anchor (φ = 0, Kd/K₀ = 1)
- **Black curve:** fitted polynomial dry-rock trend

Inspect this plot before proceeding to fluid substitution. Clean sands should
cluster at high Kd/K₀; shales and silty intervals often scatter low or negative.
The trend line should bisect the clean-sand cloud and pass through the mineral point.

In [ ]:
fig2 = fig_kd_k0(well, coeffs)
plt.show()

## 9. Fluid Substitution Functions — Step 6

Two parallel forward-substitution functions enable a direct method comparison:

### `apply_fluid_substitution` — Simm's Adaptive Approach
Applies the conditioned dry-rock trend **only to silty sands**. Clean/cemented
sands and shales use their raw inverted Kd/K₀ (the reference facies).

### `apply_default_fluid_substitution` — Baseline
Uses the **raw inverted Kd/K₀** (clipped to [0.05, 0.99]) for every facies.
This is standard Gassmann without adaptive conditioning.

**Common outputs** (suffix = `'oil'` or `'gas'`):

| Column | Quantity |
|--------|----------|
| `Ksat_{suffix}` | New saturated bulk modulus (GPa) |
| `rho_{suffix}` | Updated density (brine mass replaced) |
| `Vp_{suffix}`, `Vs_{suffix}` | New velocities — note μ is fluid-independent |
| `PR_{suffix}`, `AI_{suffix}` | Poisson's ratio and acoustic impedance |

### Execute: Apply Fluid Substitutions

Run both the Simm conditioned method and the default method for oil and gas.
Then print mean velocity and Poisson's ratio shifts for the sand facies, plus
per-facies differences between the two methods — non-zero differences appear
only in the silty-sand facies where conditioning is applied.

In [ ]:
print("[5] Applying fluid substitutions ...")

# Simm conditioned substitution
well = apply_fluid_substitution(well, coeffs, K_OIL,  RHO_OIL,  suffix='oil')
well = apply_fluid_substitution(well, coeffs, K_GAS,  RHO_GAS,  suffix='gas')

# Default substitution (raw Kd/K0 — for comparison)
well = apply_default_fluid_substitution(well, K_OIL, RHO_OIL, suffix='oil')
well = apply_default_fluid_substitution(well, K_GAS, RHO_GAS, suffix='gas')

sand = well[well['facies'].isin(['Clean Sand','Cemented Sand','Silty Sand 1','Silty Sand 2'])]
print(f"  Mean ΔVp  brine→oil (sand): {(sand['Vp_oil']  - sand['Vp']).mean():.3f} km/s")
print(f"  Mean ΔVp  brine→gas (sand): {(sand['Vp_gas']  - sand['Vp']).mean():.3f} km/s")
print(f"  Mean ΔPR  brine→oil (sand): {(sand['PR_oil']  - sand['PR']).mean():.3f}")
print(f"  Mean ΔPR  brine→gas (sand): {(sand['PR_gas']  - sand['PR']).mean():.3f}")
print()
print("  Default vs Simm differences (by facies):")
for fac in ['Clean Sand','Cemented Sand','Silty Sand 1','Silty Sand 2','Silty Shale']:
    grp = well[well['facies'] == fac]
    if len(grp) == 0: continue
    dvp_oil = (grp['Vp_default_oil'] - grp['Vp_oil']).mean()
    dvp_gas = (grp['Vp_default_gas'] - grp['Vp_gas']).mean()
    dpr_oil = (grp['PR_default_oil'] - grp['PR_oil']).mean()
    print(f"    {fac}: ΔVp_oil={dvp_oil:.3f}, ΔVp_gas={dvp_gas:.3f}, ΔPR_oil={dpr_oil:.3f}")

### 3a. Figure 1b — Zoomed Well Log (2120–2320 m)

8-track display restricted to the Heimdal reservoir interval, showing brine/oil/gas substituted logs and Top Heimdal (2153 m) / OWC (2183 m) markers.

In [ ]:
fig1b = fig_well_logs_zoomed(well)
plt.show()

### 3b. Kd/K₀ vs Porosity — Classified Sections

Two views of the dry-frame template restricted to **labelled facies only** (Background excluded):
- **Left:** coloured by facies class
- **Right:** coloured continuously by **Vsh** — tests whether facies boundaries align with the shale-volume gradient

In [ ]:
# Kd/K0 vs porosity — classified sections only, two colour schemes
classified = well[well['facies'] != 'Background']
valid = classified[(classified['Kd_K0'] > -0.35) & (classified['Kd_K0'] < 1.05)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax in (ax1, ax2):
    for c in [0.1, 0.2, 0.3, 0.4, 0.5]:
        kd = c / (1.0 + c)
        ax.axhline(kd, color='#888888', ls='--', lw=0.9, alpha=0.7)
        ax.text(0.505, kd + 0.007, f'Kφ/K₀={c:.1f}', fontsize=7, color='#555555', va='bottom')
    ax.axhline(0.0, color='black', lw=0.8, ls=':')
    ax.set_xlim(-0.01, 0.52); ax.set_ylim(-0.35, 1.08)
    ax.set_xlabel('Effective Porosity  φ_e  (v/v)', fontsize=11)
    ax.set_ylabel('K_d / K₀', fontsize=11)
    ax.grid(True, alpha=0.25)

# Left: coloured by facies
for fac in ['Cemented Sand', 'Silty Sand 2', 'Silty Sand 1', 'Silty Shale', 'Clean Sand']:
    grp = valid[valid['facies'] == fac]
    ax1.scatter(grp['phi'], grp['Kd_K0'],
                c=FACIES_COLORS.get(fac, '#999999'),
                s=14, alpha=0.70, edgecolors='none', label=fac)
ax1.scatter([0.0], [1.0], marker='*', s=200, c='black', zorder=7, label='Mineral point')
ax1.set_title('Classified sections — coloured by facies', fontsize=11)
ax1.legend(fontsize=8, markerscale=1.5)

# Right: coloured by Vsh
sc = ax2.scatter(valid['phi'], valid['Kd_K0'],
                 c=valid['Vsh'], cmap='RdYlGn_r', vmin=0, vmax=1,
                 s=14, alpha=0.70, edgecolors='none')
plt.colorbar(sc, ax=ax2, label='Vsh  (v/v)', shrink=0.85)
ax2.scatter([0.0], [1.0], marker='*', s=200, c='black', zorder=7, label='Mineral point')
ax2.legend(fontsize=8)
ax2.set_title('Classified sections — coloured by Vsh', fontsize=11)

fig.suptitle('Kd/K₀ vs Porosity — Classified Sections  (Background excluded)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Figure 1 — Full Well Log Display (8-Track)

With the fluid substitution complete, we can now display the full log suite
showing all three fluid scenarios side by side. The **eight tracks** are:

1. **Depth ruler** — depth labels in metres
2. **GR** — gamma-ray, the primary shale indicator (GAPI)
3. **Vsh** — shale volume derived from GR normalisation
4. **φ_e** — effective porosity from density log
5. **ρ** — bulk density
6. **Vp** — P-wave velocity for all three fluids + both methods
7. **Vs** — S-wave velocity (same colour scheme)
8. **Poisson's ratio** — the most sensitive fluid indicator (reference line at σ = 0.33)

Colour-shaded facies bands and Top Heimdal / OWC markers give geological context.

In [ ]:
fig1 = fig_well_logs(well)
plt.show()

## 11. Figure 1b — Zoomed Well Log (2120–2320 m)

Same 8-track layout restricted to the **reservoir interval**, with 10 m depth
labels instead of 50 m. This zoom resolves the fluid-substitution response
across the Heimdal Formation oil column and the water leg below the OWC (2183 m).

In [ ]:
fig1b = fig_well_logs_zoomed(well)
plt.show()

## 12. Figure 3 — Rock Physics Crossplot Suite

Four complementary crossplots, each coloured by facies and showing brine (circles),
oil-substituted (triangles), and gas-substituted (squares) populations:

| Panel | Axes | Purpose |
|-------|------|---------|
| (a) | Vp vs φ_e | Velocity-porosity; cement vs sorting effect |
| (b) | Vp/Vs vs Vp | Combines lithology and fluid sensitivity |
| (c) | AI vs Vp/Vs | Standard AVO proxy — gas sands move lower-left |
| (d) | Kd/K₀ vs φ_e coloured by Vsh | Dry-rock frame; fluid-independent |

Together these four panels characterise the rock physics response before and
after substitution, and highlight which facies are most sensitive to fluid changes.

In [ ]:
fig3 = fig_crossplots(well)
plt.show()

## 13. Figure 4 — Fluid Substitution Sensitivity Arrows

Focussing on **sand facies only**, this figure draws arrows from each brine
sample to its oil- and gas-substituted counterpart:

- **Panel (a): AI vs Vp/Vs** — the AVO template. Gas substitution drives sands
  toward lower AI and higher Vp/Vs (class II/III gas sand territory). Oil shows
  a smaller but directionally similar shift.
- **Panel (b): AI vs Poisson's ratio** — the seismic inversion plane. Gas sands
  move sharply toward lower σ.

The spread of arrows by facies colour reveals which lithologies are most
sensitive to fluid changes — clean porous sands show the largest vectors.

In [ ]:
fig4 = fig_fluid_sensitivity(well)
plt.show()

## 14. Figure 5 — Per-Facies Vp: Default vs Simm Conditioning

One depth-profile panel per sand/silty class. Each panel overlays five Vp curves:

| Style | Meaning |
|-------|---------|
| Blue solid | Observed brine Vp |
| Red dashed | Default oil (raw Kd/K₀) |
| Green solid | Simm oil (conditioned Kd/K₀) |
| Magenta dotted | Default gas |
| Cyan solid | Simm gas |

For clean and cemented sands the methods are identical (they share the same raw
Kd/K₀). Differences appear only in the silty-sand facies where conditioning is applied.

In [ ]:
fig5 = fig_facies_comparison(well)
plt.show()

## 15. Figure 6 — Quantifying the Impact of Dry-Rock Conditioning

Four difference crossplots (Default minus Simm) to show **where and how much**
the adaptive dry-rock trend changes the predicted fluid substitution:

| Panel | Quantity |
|-------|----------|
| (a) | ΔVp_oil vs porosity |
| (b) | ΔVp_gas vs porosity |
| (c) | ΔVp_oil vs Vsh |
| (d) | ΔPR_oil vs porosity |

Points above zero mean the default method predicts **higher** values than Simm's
conditioned approach. Silty sands (where conditioning is applied) will show the
largest deviations; clean sands and shales cluster near zero.

In [ ]:
fig6 = fig_default_vs_simm_differences(well)
plt.show()